In [1]:
# CELL 1 — Imports and Device

import os
import json
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("FUSION LAYER DEVICE INFORMATION")
print("=" * 60)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("Using device    :", device)

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)

FUSION LAYER DEVICE INFORMATION
PyTorch version : 2.0.1+cu118
CUDA available  : True
Using device    : cuda
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
CUDA version    : 11.8


In [2]:
# CELL 2 — Project Paths & File Verification

from pathlib import Path

# Repo root: walk up from the notebook's directory until app.py is found.
BASE_PATH = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "app.py").is_file())

DATASETS_PATH = os.path.join(
    BASE_PATH,
    "datasets"
)

MODELS_PATH = os.path.join(
    BASE_PATH,
    "models"
)

RESULTS_PATH = os.path.join(
    BASE_PATH,
    "results"
)

# Create directories if they don't already exist
os.makedirs(DATASETS_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)


print("=" * 60)
print("FUSION LAYER PROJECT PATHS")
print("=" * 60)

print(f"Base path     : {BASE_PATH}")
print(f"Datasets path : {DATASETS_PATH}")
print(f"Models path   : {MODELS_PATH}")
print(f"Results path  : {RESULTS_PATH}")


# ------------------------------------------------------------
# Check required datasets
# ------------------------------------------------------------

required_datasets = [
    "alzheimer.csv",
    "nep499.csv"
]

print("\nChecking required datasets:")
print("-" * 60)

for filename in required_datasets:

    filepath = os.path.join(
        DATASETS_PATH,
        filename
    )

    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024

        print(
            f"✅ {filename:<20} "
            f"({size_kb:.2f} KB)"
        )

    else:
        print(
            f"❌ {filename:<20} "
            f"NOT FOUND"
        )


# ------------------------------------------------------------
# Check required model files
# ------------------------------------------------------------

print("\nChecking model directory:")
print("-" * 60)

if os.path.exists(MODELS_PATH):

    model_files = os.listdir(MODELS_PATH)

    if len(model_files) == 0:
        print("⚠️ Models directory is empty.")

    else:
        for filename in sorted(model_files):
            print(f"   {filename}")

else:
    print("❌ Models directory does not exist.")


print("\n" + "=" * 60)
print("PATH VERIFICATION COMPLETE")
print("=" * 60)

FUSION LAYER PROJECT PATHS
Base path     : .
Datasets path : datasets
Models path   : models
Results path  : results

Checking required datasets:
------------------------------------------------------------
✅ alzheimer.csv        (16.06 KB)
✅ nep499.csv           (26.78 KB)

Checking model directory:
------------------------------------------------------------
   efficientnet_b0_best.pth
   efficientnet_b4_best.pth
   efficientnet_b4_info.json
   ensemble_info.json
   ensemble_meta_learner.pkl
   fusion_layer_best.pth
   fusion_scaler.pkl
   meta_learner.pkl
   resnet152_best.pth
   resnet50_best.pth
   vgg16_best.pth
   vit_best.pth

PATH VERIFICATION COMPLETE


In [3]:
# CELL 3 — Load Clinical and Genetic Data

alzheimer_df = pd.read_csv(
    os.path.join(
        DATASETS_PATH,
        "alzheimer.csv"
    )
)

gene_df = pd.read_csv(
    os.path.join(
        DATASETS_PATH,
        "nep499.csv"
    )
)

print("=" * 60)
print("CLINICAL DATA")
print("=" * 60)

print("Samples:", len(alzheimer_df))
print("Columns:", list(alzheimer_df.columns))

print("\nGroup distribution:")
print(
    alzheimer_df["Group"]
    .value_counts()
)


print("\n" + "=" * 60)
print("GENETIC DATA")
print("=" * 60)

print("Samples:", len(gene_df))
print("Columns:", list(gene_df.columns))

print("\nStatus distribution:")
print(
    gene_df["status"]
    .value_counts()
)


print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print("\nClinical:")
print(
    alzheimer_df.isna()
    .sum()
)

print("\nGenetic:")
print(
    gene_df.isna()
    .sum()
)

CLINICAL DATA
Samples: 373
Columns: ['Group', 'M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']

Group distribution:
Group
Nondemented    190
Demented       146
Converted       37
Name: count, dtype: int64

GENETIC DATA
Samples: 499
Columns: ['Unnamed: 0', 'id', 'sex', 'age', 'APOE.a1', 'APOE.a2', 'apoe4', 'status', 'R6.a1', 'R6.a2', 'N4.a1', 'N4.a2', 'N6.a1', 'N6.a2', 'N11.a1', 'N11.a2', 'N15.a1', 'N15.a2', 'N18.a1', 'N18.a2', 'N22.a1', 'N22.a2', 'N24.a1', 'N24.a2']

Status distribution:
status
1    257
0    242
Name: count, dtype: int64

MISSING VALUES

Clinical:
Group     0
M/F       0
Age       0
EDUC      0
SES      19
MMSE      2
CDR       0
eTIV      0
nWBV      0
ASF       0
dtype: int64

Genetic:
Unnamed: 0    0
id            0
sex           0
age           0
APOE.a1       0
APOE.a2       0
apoe4         0
status        0
R6.a1         0
R6.a2         0
N4.a1         0
N4.a2         0
N6.a1         0
N6.a2         0
N11.a1        0
N11.a2        0
N15.a1      

In [4]:
# CELL 4 — Create Diagnosis Labels

def create_diagnosis_label(group):

    group = str(group).strip()

    if group in ["Demented", "Converted"]:
        return 0

    if group == "Nondemented":
        return 1

    return np.nan


alzheimer_df["diagnosis"] = (
    alzheimer_df["Group"]
    .apply(create_diagnosis_label)
)

print("=" * 60)
print("DIAGNOSIS LABEL VERIFICATION")
print("=" * 60)

print("\nMapping:")
print("Demented     -> 0")
print("Converted    -> 0")
print("Nondemented  -> 1")

print("\nDiagnosis distribution:")
print(
    alzheimer_df["diagnosis"]
    .value_counts()
    .sort_index()
)

print("\nMapping verification:")

for group in alzheimer_df["Group"].unique():

    label = create_diagnosis_label(group)

    count = (
        alzheimer_df["Group"] == group
    ).sum()

    print(
        f"{group:<15} "
        f"count={count:<3} "
        f"label={label}"
    )

print(
    "\nMissing diagnosis labels:",
    alzheimer_df["diagnosis"].isna().sum()
)

DIAGNOSIS LABEL VERIFICATION

Mapping:
Demented     -> 0
Converted    -> 0
Nondemented  -> 1

Diagnosis distribution:
diagnosis
0    183
1    190
Name: count, dtype: int64

Mapping verification:
Nondemented     count=190 label=1
Demented        count=146 label=0
Converted       count=37  label=0

Missing diagnosis labels: 0


In [5]:
# CELL 5 — Clinical Features

clinical_features = alzheimer_df[
    [
        "MMSE",
        "CDR",
        "nWBV",
        "diagnosis"
    ]
].copy()

clinical_features["MMSE"] = (
    clinical_features["MMSE"]
    .fillna(
        clinical_features["MMSE"].median()
    )
)

clinical_features["CDR"] = (
    clinical_features["CDR"]
    .fillna(
        clinical_features["CDR"].median()
    )
)

clinical_features["nWBV"] = (
    clinical_features["nWBV"]
    .fillna(
        clinical_features["nWBV"].median()
    )
)

clinical_features["mmse_norm"] = (
    clinical_features["MMSE"] / 30.0
)

clinical_scaler = StandardScaler()

clinical_features["cdr_norm"] = (
    clinical_scaler.fit_transform(
        clinical_features[["CDR"]]
    ).ravel()
)

clinical_features["nwbv_norm"] = (
    clinical_scaler.fit_transform(
        clinical_features[["nWBV"]]
    ).ravel()
)


print("=" * 60)
print("CLINICAL FEATURES")
print("=" * 60)

print("Samples:", len(clinical_features))

print("\nFeatures:")
print(
    clinical_features[
        [
            "mmse_norm",
            "cdr_norm",
            "nwbv_norm"
        ]
    ].head()
)

print("\nFeature statistics:")

print(
    clinical_features[
        [
            "mmse_norm",
            "cdr_norm",
            "nwbv_norm"
        ]
    ].describe()
)

print(
    "\nMissing values:",
    clinical_features[
        [
            "mmse_norm",
            "cdr_norm",
            "nwbv_norm"
        ]
    ].isna().sum().sum()
)

CLINICAL FEATURES
Samples: 373

Features:
   mmse_norm  cdr_norm  nwbv_norm
0   0.900000 -0.777653  -0.905169
1   1.000000 -0.777653  -1.309643
2   0.766667  0.559050   0.173429
3   0.933333  0.559050  -0.446765
4   0.733333  0.559050  -0.770344

Feature statistics:
        mmse_norm      cdr_norm     nwbv_norm
count  373.000000  3.730000e+02  3.730000e+02
mean     0.911707 -6.191056e-17 -4.762351e-16
std      0.122511  1.001343e+00  1.001343e+00
min      0.133333 -7.776529e-01 -2.307345e+00
25%      0.900000 -7.776529e-01 -7.973089e-01
50%      0.966667 -7.776529e-01 -1.532591e-02
75%      1.000000  5.590500e-01  7.127272e-01
max      1.000000  4.569159e+00  2.896887e+00

Missing values: 0


In [6]:
# CELL 6 — APOE Features

def get_apoe_status(row):

    a1 = int(row["APOE.a1"])
    a2 = int(row["APOE.a2"])

    if a1 > a2:
        a1, a2 = a2, a1

    return f"E{a1}/E{a2}"


apoe_encoding = {
    "E2/E2": [1, 0, 0, 0, 0, 0],
    "E2/E3": [0, 1, 0, 0, 0, 0],
    "E3/E3": [0, 0, 1, 0, 0, 0],
    "E2/E4": [0, 0, 0, 1, 0, 0],
    "E3/E4": [0, 0, 0, 0, 1, 0],
    "E4/E4": [0, 0, 0, 0, 0, 1]
}


gene_df["apoe_status"] = gene_df.apply(
    get_apoe_status,
    axis=1
)

gene_df["apoe_encoded"] = (
    gene_df["apoe_status"]
    .map(apoe_encoding)
)


print("=" * 60)
print("APOE FEATURES")
print("=" * 60)

print("Samples:", len(gene_df))

print("\nAPOE distribution:")
print(
    gene_df["apoe_status"]
    .value_counts()
)

print("\nSample encodings:")

for _, row in gene_df.head(5).iterrows():

    print(
        f"{row['apoe_status']:<8} "
        f"-> {row['apoe_encoded']}"
    )

print(
    "\nMissing APOE encodings:",
    gene_df["apoe_encoded"].isna().sum()
)

APOE FEATURES
Samples: 499

APOE distribution:
apoe_status
E3/E3    305
E2/E3     88
E3/E4     76
E2/E4     16
E4/E4      8
E2/E2      6
Name: count, dtype: int64

Sample encodings:
E2/E3    -> [0, 1, 0, 0, 0, 0]
E3/E4    -> [0, 0, 0, 0, 1, 0]
E3/E3    -> [0, 0, 1, 0, 0, 0]
E3/E3    -> [0, 0, 1, 0, 0, 0]
E3/E3    -> [0, 0, 1, 0, 0, 0]

Missing APOE encodings: 0


In [7]:
# CELL 7 — Check Clinical/Genetic Matching

clinical_match = alzheimer_df[
    ["M/F", "Age"]
].copy()

genetic_match = gene_df[
    ["sex", "age"]
].copy()

clinical_match["sex"] = (
    clinical_match["M/F"]
    .astype(str)
    .str.upper()
    .str.strip()
)

genetic_match["sex"] = (
    genetic_match["sex"]
    .astype(str)
    .str.upper()
    .str.strip()
)

clinical_match["age"] = pd.to_numeric(
    clinical_match["Age"],
    errors="coerce"
)

genetic_match["age"] = pd.to_numeric(
    genetic_match["age"],
    errors="coerce"
)

print("=" * 60)
print("CLINICAL / GENETIC MATCHING CHECK")
print("=" * 60)

print("Clinical records:", len(clinical_match))
print("Genetic records :", len(genetic_match))

print("\nClinical sex values:")
print(clinical_match["sex"].value_counts())

print("\nGenetic sex values:")
print(genetic_match["sex"].value_counts())

print("\nExact age + sex matches:")

matches = clinical_match.merge(
    genetic_match,
    on=["sex", "age"],
    how="inner"
)

print("Matched rows:", len(matches))

print(
    "Unique clinical records matched:",
    matches[["sex", "age"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nClinical age range:",
    clinical_match["age"].min(),
    "-",
    clinical_match["age"].max()
)

print(
    "Genetic age range:",
    genetic_match["age"].min(),
    "-",
    genetic_match["age"].max()
)

CLINICAL / GENETIC MATCHING CHECK
Clinical records: 373
Genetic records : 499

Clinical sex values:
sex
F    213
M    160
Name: count, dtype: int64

Genetic sex values:
sex
2    281
1    218
Name: count, dtype: int64

Exact age + sex matches:
Matched rows: 0
Unique clinical records matched: 0

Clinical age range: 60 - 98
Genetic age range: 58 - 99


In [8]:
# CELL 8 — Check Possible Patient Identifiers

print("=" * 60)
print("PATIENT IDENTIFIER CHECK")
print("=" * 60)

print("\nClinical columns:")
for col in alzheimer_df.columns:
    print(f"  {col}")

print("\nGenetic columns:")
for col in gene_df.columns:
    print(f"  {col}")

print("\nPotential identifier columns:")

clinical_candidates = [
    col for col in alzheimer_df.columns
    if any(
        key in col.lower()
        for key in ["id", "subject", "patient", "participant"]
    )
]

genetic_candidates = [
    col for col in gene_df.columns
    if any(
        key in col.lower()
        for key in ["id", "subject", "patient", "participant"]
    )
]

print(
    "Clinical:",
    clinical_candidates
)

print(
    "Genetic:",
    genetic_candidates
)

if "id" in gene_df.columns:

    print("\nGenetic ID examples:")
    print(
        gene_df["id"]
        .head(10)
        .tolist()
    )

print("\nClinical index examples:")
print(
    alzheimer_df.index[:10].tolist()
)

PATIENT IDENTIFIER CHECK

Clinical columns:
  Group
  M/F
  Age
  EDUC
  SES
  MMSE
  CDR
  eTIV
  nWBV
  ASF
  diagnosis

Genetic columns:
  Unnamed: 0
  id
  sex
  age
  APOE.a1
  APOE.a2
  apoe4
  status
  R6.a1
  R6.a2
  N4.a1
  N4.a2
  N6.a1
  N6.a2
  N11.a1
  N11.a2
  N15.a1
  N15.a2
  N18.a1
  N18.a2
  N22.a1
  N22.a2
  N24.a1
  N24.a2
  apoe_status
  apoe_encoded

Potential identifier columns:
Clinical: []
Genetic: ['id']

Genetic ID examples:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Clinical index examples:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [9]:
# CELL 9 — Inspect MRI Subject Identifiers

MRI_DATA_PATH = DATASETS_PATH / "KAGGLE DATASET" / "Combined Dataset" / "train"

print("=" * 60)
print("MRI / CLINICAL SUBJECT RELATIONSHIP")
print("=" * 60)

print("\nClinical dataset:")
print("Rows:", len(alzheimer_df))

print("\nClinical columns:")
print(list(alzheimer_df.columns))

print("\nFirst clinical rows:")
print(
    alzheimer_df[
        ["Group", "M/F", "Age", "MMSE", "CDR", "nWBV"]
    ].head(10)
)


print("\nMRI dataset path:")
print(MRI_DATA_PATH)

if os.path.exists(MRI_DATA_PATH):

    class_folders = sorted(
        [
            name
            for name in os.listdir(MRI_DATA_PATH)
            if os.path.isdir(
                os.path.join(
                    MRI_DATA_PATH,
                    name
                )
            )
        ]
    )

    print("\nMRI classes:")
    print(class_folders)

    print("\nSample MRI filenames:")

    shown = 0

    for class_name in class_folders:

        class_path = os.path.join(
            MRI_DATA_PATH,
            class_name
        )

        files_in_class = sorted(
            os.listdir(class_path)
        )

        for filename in files_in_class[:5]:

            print(
                f"{class_name} -> {filename}"
            )

        shown += min(5, len(files_in_class))

else:

    print("\nMRI dataset path not found.")

MRI / CLINICAL SUBJECT RELATIONSHIP

Clinical dataset:
Rows: 373

Clinical columns:
['Group', 'M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF', 'diagnosis']

First clinical rows:
         Group M/F  Age  MMSE  CDR   nWBV
0  Nondemented   M   87  27.0  0.0  0.696
1  Nondemented   M   88  30.0  0.0  0.681
2     Demented   M   75  23.0  0.5  0.736
3     Demented   M   76  28.0  0.5  0.713
4     Demented   M   80  22.0  0.5  0.701
5  Nondemented   F   88  28.0  0.0  0.710
6  Nondemented   F   90  27.0  0.0  0.718
7  Nondemented   M   80  28.0  0.0  0.712
8  Nondemented   M   83  29.0  0.5  0.711
9  Nondemented   M   85  30.0  0.0  0.705

MRI dataset path:
datasets\KAGGLE DATASET\Combined Dataset\train

MRI classes:
['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']

Sample MRI filenames:
Mild Impairment -> MildImpairment (1).jpg
Mild Impairment -> MildImpairment (10).jpg
Mild Impairment -> MildImpairment (100).jpg
Mild Impairment -> MildImp

In [10]:
# CELL 9 — Clinical Fusion Features

clinical_X = clinical_features[
    [
        "mmse_norm",
        "cdr_norm",
        "nwbv_norm"
    ]
].values.astype(np.float32)

clinical_y = (
    clinical_features["diagnosis"]
    .values.astype(np.int64)
)

print("Clinical features:", clinical_X.shape)
print("Clinical labels:", clinical_y.shape)
print("Classes:", np.unique(clinical_y))

Clinical features: (373, 3)
Clinical labels: (373,)
Classes: [0 1]


In [11]:
# CELL 10 — Clinical Fusion Network

class ClinicalFusionNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)


X_tensor = torch.tensor(
    clinical_X,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    clinical_y,
    dtype=torch.long
)

dataset = TensorDataset(
    X_tensor,
    y_tensor
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_set, val_set = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False
)

clinical_model = ClinicalFusionNet().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    clinical_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("Training device:", device)
print("Train samples:", len(train_set))
print("Validation samples:", len(val_set))
print("Parameters:", sum(
    p.numel()
    for p in clinical_model.parameters()
))

Training device: cuda
Train samples: 298
Validation samples: 75
Parameters: 690


In [12]:
# CELL 11 — Train Clinical Branch

EPOCHS = 50

best_val_acc = 0.0

for epoch in range(EPOCHS):

    clinical_model.train()

    train_correct = 0
    train_total = 0
    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = clinical_model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == y_batch
        ).sum().item()

        train_total += y_batch.size(0)

    clinical_model.eval()

    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = clinical_model(X_batch)
            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == y_batch
            ).sum().item()

            val_total += y_batch.size(0)

    train_acc = (
        100 * train_correct / train_total
    )

    val_acc = (
        100 * val_correct / val_total
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            clinical_model.state_dict(),
            os.path.join(
                MODELS_PATH,
                "clinical_branch_best.pth"
            )
        )

        saved = " Saved"

    else:
        saved = ""

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss / len(train_loader):.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss / len(val_loader):.4f} | "
        f"Val Acc: {val_acc:.2f}%{saved}"
    )

print(
    f"\nBest validation accuracy: "
    f"{best_val_acc:.2f}%"
)

Epoch 1/50 | Train Loss: 0.7013 | Train Acc: 41.95% | Val Loss: 0.6653 | Val Acc: 66.67% Saved
Epoch 2/50 | Train Loss: 0.6727 | Train Acc: 61.07% | Val Loss: 0.6322 | Val Acc: 86.67% Saved
Epoch 3/50 | Train Loss: 0.6422 | Train Acc: 73.49% | Val Loss: 0.5992 | Val Acc: 93.33% Saved
Epoch 4/50 | Train Loss: 0.6104 | Train Acc: 83.56% | Val Loss: 0.5618 | Val Acc: 96.00% Saved
Epoch 5/50 | Train Loss: 0.5745 | Train Acc: 86.58% | Val Loss: 0.5196 | Val Acc: 96.00%
Epoch 6/50 | Train Loss: 0.5269 | Train Acc: 90.94% | Val Loss: 0.4736 | Val Acc: 97.33% Saved
Epoch 7/50 | Train Loss: 0.4956 | Train Acc: 88.93% | Val Loss: 0.4232 | Val Acc: 97.33%
Epoch 8/50 | Train Loss: 0.4626 | Train Acc: 90.94% | Val Loss: 0.3716 | Val Acc: 97.33%
Epoch 9/50 | Train Loss: 0.4076 | Train Acc: 91.95% | Val Loss: 0.3255 | Val Acc: 97.33%
Epoch 10/50 | Train Loss: 0.3673 | Train Acc: 92.62% | Val Loss: 0.2797 | Val Acc: 97.33%
Epoch 11/50 | Train Loss: 0.3182 | Train Acc: 92.95% | Val Loss: 0.2421 | Val A

In [13]:
# CELL 12 — Save Clinical Branch

CLINICAL_MODEL_PATH = os.path.join(
    MODELS_PATH,
    "clinical_branch_best.pth"
)

CLINICAL_INFO_PATH = os.path.join(
    MODELS_PATH,
    "clinical_branch_info.json"
)

torch.save(
    clinical_model.state_dict(),
    CLINICAL_MODEL_PATH
)

clinical_info = {
    "model": "ClinicalFusionNet",
    "features": [
        "MMSE",
        "CDR",
        "nWBV"
    ],
    "input_features": 3,
    "classes": {
        "0": "Demented/Converted",
        "1": "Nondemented"
    },
    "train_samples": len(train_set),
    "validation_samples": len(val_set),
    "best_validation_accuracy": float(best_val_acc)
}

with open(
    CLINICAL_INFO_PATH,
    "w"
) as f:
    json.dump(
        clinical_info,
        f,
        indent=4
    )

print("Clinical branch saved.")
print("Model:", CLINICAL_MODEL_PATH)
print("Info :", CLINICAL_INFO_PATH)
print(
    f"Best validation accuracy: "
    f"{best_val_acc:.2f}%"
)

Clinical branch saved.
Model: models\clinical_branch_best.pth
Info : models\clinical_branch_info.json
Best validation accuracy: 97.33%
